# Avance 2 - Estructura y Calidad de los Datos (Sección 3.1)
**Asignatura:** EIN092B Visualización de Datos  
**Profesor:** Jesús A. Parra  
**Proyecto:** Impacto de las Áreas Verdes en la Tasa de Hurtos (Región Metropolitana, 2023)  
**Integrantes:** Vicente Paredes, Tomás Armijo, Francisca Matamala  

---

## 🎯 Objetivo de este Notebook
En cumplimiento con la **Sección 3.1** de la pauta del Avance 2 (*Estructura y calidad de los datos*), este notebook documenta el proceso reproducible de:
1. **Auditoría de datos crudos:** Inspección de dimensiones, tipos de variables y valores faltantes en las 3 fuentes (MINVU, CEAD y SAE).
2. **Justificación de limpieza y filtros:** 
   - Horizonte temporal: año 2023 ($T$).
   - Alcance territorial: 52 comunas de la Región Metropolitana.
   - Tipo de delito: delimitado a *Hurtos* ($Y$).
3. **Integración territorial:** Unión mediante el Código Único Territorial (`CUT`) oficial.
4. **Ingeniería de variables normalizadas:** Resolución del sesgo de población mediante:
   - $X$: Metros cuadrados de áreas verdes por habitante ($m^2/\text{hab}$).
   - $Y$: Tasa de hurtos por cada 100.000 habitantes.
   - Variable de control contextual: Porcentaje de pobreza por ingresos.
5. **Exportación:** Guardado de los datasets limpios en `data/processed/` para el análisis y la aplicación en Streamlit.

In [1]:
import os
import pandas as pd
import numpy as np

# Configuración de visualización en pandas
pd.set_option('display.max_columns', 25)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Definición de rutas relativas del repositorio
RAW_DIR = os.path.join('..', 'data', 'raw')
PROC_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)

print('Directorio de datos crudos:', os.path.abspath(RAW_DIR))
print('Directorio de datos procesados:', os.path.abspath(PROC_DIR))

Directorio de datos crudos: /home/r0ars/Documents/Impacto-de-las-Areas-Verdes-en-la-Tasa-de-Hurtos-2023/data/raw
Directorio de datos procesados: /home/r0ars/Documents/Impacto-de-las-Areas-Verdes-en-la-Tasa-de-Hurtos-2023/data/processed


---
## 1. Dataset Áreas Verdes (IDE MINVU - INE)
* **Archivo:** `Areas_Verdes_Minvu_INE.csv`
* **Unidad de observación cruda:** Un espacio público catastrado (plaza o parque).
* **Cobertura:** El catastro MINVU registra áreas verdes en **zonas urbanas** (`AREA_C = URBANO`).
* **Objetivo de limpieza:** Filtrar las áreas verdes de la Región Metropolitana y agregarlas a nivel comunal (`CUT`).

In [2]:
# 1.1 Carga e inspección inicial
path_av = os.path.join(RAW_DIR, 'Areas_Verdes_Minvu_INE.csv')
df_av_raw = pd.read_csv(path_av, low_memory=False)
print(f'Dimensiones crudas a nivel nacional: {df_av_raw.shape[0]} filas, {df_av_raw.shape[1]} columnas.')

# 1.2 Filtro territorial: Región Metropolitana (códigos CUT inician con '13')
df_av_rm = df_av_raw[df_av_raw['CUT'].astype(str).str.startswith('13')].copy()
print(f'Áreas verdes pertenecientes a la Región Metropolitana: {len(df_av_rm)} registros.')
print(f'Superficie total en la RM: {df_av_rm["Shape_Area"].sum():,.2f} m².')

# 1.3 Agregación a nivel comunal
df_av_comunal = df_av_rm.groupby('CUT', as_index=False).agg(
    m2_areas_verdes=('Shape_Area', 'sum'),
    cantidad_espacios_publicos=('OBJECTID_1', 'count'),
    cantidad_plazas=('TIPO_EP', lambda s: (s == 'PLAZA').sum()),
    cantidad_parques=('TIPO_EP', lambda s: (s == 'PARQUE').sum())
)
df_av_comunal['CUT'] = df_av_comunal['CUT'].astype(int)

print(f'Total comunas con áreas verdes catastradas: {len(df_av_comunal)}')
df_av_comunal.head()

Dimensiones crudas a nivel nacional: 18173 filas, 20 columnas.
Áreas verdes pertenecientes a la Región Metropolitana: 8160 registros.
Superficie total en la RM: 38,412,044.44 m².
Total comunas con áreas verdes catastradas: 44


,CUT,m2_areas_verdes,cantidad_espacios_publicos,cantidad_plazas,cantidad_parques
0,13101,1817641.552,117,105,12
1,13102,1235045.249,138,114,24
2,13103,1198693.962,133,113,20
3,13104,382947.550,169,152,17
4,13105,333170.549,124,117,7


---
## 2. Dataset Delictual (CEAD - Subsecretaría de Prevención del Delito)
* **Archivo:** `cead_delincuencia_chile.parquet`
* **Unidad de observación cruda:** Registros mensuales de casos policiales por tipo de delito y comuna.
* **Objetivo de limpieza:** Filtrar el año 2023 ($T$), la Región Metropolitana y acotar al delito de *Hurtos* ($Y$).

In [3]:
# 2.1 Carga e inspección inicial
path_cead = os.path.join(RAW_DIR, 'cead_delincuencia_chile.parquet')
df_cead_raw = pd.read_parquet(path_cead)
print(f'Dimensiones crudas históricas: {df_cead_raw.shape[0]:,} filas, {df_cead_raw.shape[1]} columnas.')

# 2.2 Filtro temporal (2023), regional (RM) y delictual (Hurtos)
df_cead_raw['fecha'] = pd.to_datetime(df_cead_raw['fecha'])
df_cead_rm_2023 = df_cead_raw[
    (df_cead_raw['fecha'].dt.year == 2023) & 
    (df_cead_raw['cut_region'] == 13.0) & 
    (df_cead_raw['delito'] == 'Hurtos')
].copy()

total_casos_hurtos = df_cead_rm_2023['delito_n'].sum()
print(f'Total de casos policiales por Hurtos en la RM durante 2023: {int(total_casos_hurtos):,}')

# 2.3 Agregación anual comunal
df_hurtos_comunal = df_cead_rm_2023.groupby('cut_comuna', as_index=False).agg(
    total_hurtos_2023=('delito_n', 'sum')
)
df_hurtos_comunal['CUT'] = df_hurtos_comunal['cut_comuna'].astype(int)
df_hurtos_comunal = df_hurtos_comunal[['CUT', 'total_hurtos_2023']]

# 2.4 Serie mensual para el análisis temporal T
df_hurtos_mensual = df_cead_rm_2023.groupby(['cut_comuna', 'comuna', 'fecha'], as_index=False).agg(
    hurtos=('delito_n', 'sum')
)
df_hurtos_mensual['CUT'] = df_hurtos_mensual['cut_comuna'].astype(int)
df_hurtos_mensual['mes'] = df_hurtos_mensual['fecha'].dt.month
df_hurtos_comunal.head()

Dimensiones crudas históricas: 1,987,200 filas, 7 columnas.
Total de casos policiales por Hurtos en la RM durante 2023: 48,737


,CUT,total_hurtos_2023
0,13101,8038.000
1,13102,885.000
2,13103,354.000
3,13104,357.000
4,13105,547.000


---
## 3. Dataset Demográfico y Socioeconómico (SAE - Ministerio de Desarrollo Social)
* **Archivo:** `sae_ingresos_2023.xlsx` (su contenido corresponde a la publicación **SAE 2024**, según el título interno del archivo).
* **Unidad de observación:** Comuna chilena.
* **Uso en el proyecto:** Población proyectada por comuna (para normalizar indicadores) y porcentaje de pobreza por ingresos (variable de contexto).
* **Limitación:** El periodo del proyecto es 2023 y esta estimación es de 2024. Se usa como la aproximación más cercana disponible; la población comunal varía poco de un año a otro.

In [4]:
# 3.1 Carga del dataset (omitiendo encabezados institucionales)
path_sae = os.path.join(RAW_DIR, 'sae_ingresos_2023.xlsx')
df_sae_raw = pd.read_excel(path_sae, skiprows=2)

col_codigo = df_sae_raw.columns[0]
col_nombre = df_sae_raw.columns[2]
col_pob = df_sae_raw.columns[3]
col_pob_pct = df_sae_raw.columns[5]

# Filtrar las 52 comunas de la RM
df_sae_rm = df_sae_raw[df_sae_raw[col_codigo].astype(str).str.startswith('13')].copy()
df_sae_rm['CUT'] = df_sae_rm[col_codigo].astype(int)

df_sae_clean = df_sae_rm[['CUT', col_nombre, col_pob, col_pob_pct]].rename(columns={
    col_nombre: 'comuna',
    col_pob: 'poblacion',
    col_pob_pct: 'pct_pobreza'
})

print(f'Comunas de la RM recuperadas: {len(df_sae_clean)} comunas.')
print(f'Población total estimada de la RM: {int(df_sae_clean["poblacion"].sum()):,} habitantes.')
df_sae_clean.head()

Comunas de la RM recuperadas: 52 comunas.
Población total estimada de la RM: 8,440,142 habitantes.


,CUT,comuna,poblacion,pct_pobreza
256,13101,Santiago,547366.000,0.102
257,13102,Cerrillos,90197.000,0.176
258,13103,Cerro Navia,139237.000,0.190
259,13104,Conchalí,136237.000,0.158
260,13105,El Bosque,169555.000,0.201


---
## 4. Integración Territorial y Cálculo de Indicadores Normalizados
Unimos las 3 fuentes por `CUT` y calculamos:
1. **$X$ normalizada:** metros cuadrados de áreas verdes por habitante ($m^2/\text{hab}$).
2. **$Y$ normalizada:** tasa de hurtos por cada 100.000 habitantes.
3. **Densidad de espacios:** espacios públicos por cada 10.000 habitantes.

**Decisión de limpieza importante:** el catastro MINVU solo cubre 44 de las 52 comunas de la RM. Las 8 comunas faltantes son comunas rurales o semi-rurales (ej. Tiltil, Paine, Curacaví). Que no aparezcan en el catastro **no significa que tengan 0 m² de áreas verdes**, sino que no fueron catastradas. Por eso **no se imputan con 0**: se dejan como valor faltante (`NaN`) y se marcan con la columna `tiene_catastro_minvu`. En el análisis de la relación $X$ vs $Y$ se usarán solo las 44 comunas con catastro.

In [5]:
# 4.1 Unión (Merge) por clave comunal CUT
df_consolidado = df_sae_clean.merge(df_av_comunal, on='CUT', how='left')
df_consolidado = df_consolidado.merge(df_hurtos_comunal, on='CUT', how='left')

# Hurtos: se verifica que todas las comunas tengan dato
print('Comunas sin dato de hurtos:', df_consolidado['total_hurtos_2023'].isnull().sum())

# Áreas verdes: las comunas sin catastro MINVU se dejan como NaN (no como 0)
df_consolidado['tiene_catastro_minvu'] = df_consolidado['m2_areas_verdes'].notna()
sin_catastro = df_consolidado[~df_consolidado['tiene_catastro_minvu']]
print(f'Comunas con catastro MINVU: {df_consolidado["tiene_catastro_minvu"].sum()} de {len(df_consolidado)}')
print('Comunas sin catastro MINVU (quedan como NaN):')
print(sin_catastro[['comuna', 'poblacion', 'total_hurtos_2023']].to_string(index=False))

# 4.2 Métricas normalizadas (las comunas sin catastro quedan con NaN en las métricas de áreas verdes)
df_consolidado['m2_areas_verdes_per_capita'] = (
    df_consolidado['m2_areas_verdes'] / df_consolidado['poblacion']
)
df_consolidado['tasa_hurtos_100k'] = (
    (df_consolidado['total_hurtos_2023'] / df_consolidado['poblacion']) * 100000
)
df_consolidado['espacios_por_10k_hab'] = (
    (df_consolidado['cantidad_espacios_publicos'] / df_consolidado['poblacion']) * 10000
)

print(f'\nDimensiones de la tabla consolidada: {df_consolidado.shape[0]} comunas, {df_consolidado.shape[1]} columnas.')
df_consolidado.head()

Comunas sin dato de hurtos: 0
Comunas con catastro MINVU: 44 de 52
Comunas sin catastro MINVU (quedan como NaN):
         comuna  poblacion  total_hurtos_2023
         Tiltil  22546.000             38.000
Calera De Tango  30445.000             77.000
          Paine  88781.000            217.000
       Curacaví  38539.000            135.000
    María Pinto  15730.000             28.000
      San Pedro  12600.000             27.000
       El Monte  42437.000            109.000
  Isla De Maipo  42589.000             89.000

Dimensiones de la tabla consolidada: 52 comunas, 13 columnas.


,CUT,comuna,poblacion,pct_pobreza,m2_areas_verdes,cantidad_espacios_publicos,cantidad_plazas,cantidad_parques,total_hurtos_2023,tiene_catastro_minvu,m2_areas_verdes_per_capita,tasa_hurtos_100k,espacios_por_10k_hab
0,13101,Santiago,547366.000,0.102,1817641.552,117.000,105.000,12.000,8038.000,True,3.321,1468.487,2.138
1,13102,Cerrillos,90197.000,0.176,1235045.249,138.000,114.000,24.000,885.000,True,13.693,981.186,15.300
2,13103,Cerro Navia,139237.000,0.190,1198693.962,133.000,113.000,20.000,354.000,True,8.609,254.243,9.552
3,13104,Conchalí,136237.000,0.158,382947.550,169.000,152.000,17.000,357.000,True,2.811,262.043,12.405
4,13105,El Bosque,169555.000,0.201,333170.549,124.000,117.000,7.000,547.000,True,1.965,322.609,7.313


---
## 5. Auditoría de Calidad y Estadísticas Descriptivas
Resumen estadístico para comprobar dispersión y valores atípicos. La fila `count` muestra cuántas comunas tienen dato en cada variable (44 en las variables de áreas verdes, 52 en el resto).

In [6]:
metricas_clave = [
    'poblacion', 'pct_pobreza', 'm2_areas_verdes', 
    'm2_areas_verdes_per_capita', 'total_hurtos_2023', 'tasa_hurtos_100k'
]
df_consolidado[metricas_clave].describe()

,poblacion,pct_pobreza,m2_areas_verdes,m2_areas_verdes_per_capita,total_hurtos_2023,tasa_hurtos_100k
count,52.000,52.000,44.000,44.000,52.000,52.000
mean,162310.423,0.145,873001.010,5.095,937.250,484.103
std,141517.784,0.054,751961.717,4.359,1387.157,439.360
min,7892.000,0.008,3725.232,0.177,12.000,152.053
25%,88048.500,0.120,368913.892,2.422,257.500,256.199
50%,125235.000,0.152,551930.364,3.507,502.500,352.809
75%,190957.000,0.184,1355494.829,5.168,896.250,524.644
max,669122.000,0.233,3026512.949,16.270,8038.000,2795.063


---
## 6. Exportación de Datasets Procesados
Se exportan dos archivos limpios a `data/processed/`:
1. `comunas_rm_consolidado_2023.csv`: Las 52 comunas de la RM con indicadores listos.
2. `hurtos_mensuales_rm_2023.csv`: Serie temporal mensual 2023 para el análisis del componente $T$.

In [7]:
# 6.1 Guardar tabla comunal consolidada
path_out_comunal = os.path.join(PROC_DIR, 'comunas_rm_consolidado_2023.csv')
df_consolidado.to_csv(path_out_comunal, index=False, encoding='utf-8')
print('Dataset comunal consolidado guardado exitosamente.')

# 6.2 Guardar serie temporal mensual
path_out_mensual = os.path.join(PROC_DIR, 'hurtos_mensuales_rm_2023.csv')
df_hurtos_mensual.to_csv(path_out_mensual, index=False, encoding='utf-8')
print('Serie mensual de hurtos guardada exitosamente.')

print('\n¡Preparación completada con éxito! Datos limpios y estructurados listos para el EDA.')

Dataset comunal consolidado guardado exitosamente.
Serie mensual de hurtos guardada exitosamente.

¡Preparación completada con éxito! Datos limpios y estructurados listos para el EDA.
